In [23]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, format_document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载 arXiv 上的论文
loader = ArxivLoader(query="2210.03629", load_max_docs=1)
docs = loader.load()

# 把文本分割成200个字符一组的片段
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)

# 构建 FAISS 向量存储和对应的 Retriever
vs = FAISS.from_documents(chunks[:10], 
                          OllamaEmbeddings(model="llama2-chinese:13b", base_url="http://localhost:22515"))
vs.similarity_search("What is ReAct")
retriever = vs.as_retriever()

# 构建 Document 转文本段落的工具函数
DEFAULT_DOCUMENT_PROMPT = PromptTemplate.from_template(
    template="{page_content}"
)

def _combine_documents(
        docs, document_prompt=DEFAULT_DOCUMENT_PROMPT,
        document_seperator="\n\n"):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_seperator.join(doc_strings)

# 准备 Model I/O 三元组
template = """Answer the question using the following context:
    {context}
    
    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)
print(prompt)
model = ChatOllama(model="llama2-chinese:13b",
                   base_url="http://localhost:22515")

# 构建 RAG 链
#chain = (
#    {
#        "context": retriever | _combine_documents,
#        "question": RunnablePassthrough(),
#    }
#    | prompt
#    | model
#    | StrOutputParser()
#)

# 不使用 RAG
chain = (
    {
        # provide a Runnable that returns the constant context string
        "context": RunnableLambda(lambda *args, **kwargs: "一种软件设计方法"),
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke("什么是 ReAct？")

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question using the following context:\n    {context}\n\n    Question: {question}\n    '), additional_kwargs={})]


'ReAct（Reactive Application Architecture）是一种软件设计方法，它将应用程序分解成小型独立模块，每个模块都具有自己的状态管理和事件处理逻辑。这种分布式设计方法可以提高应用程序的可维护性、可扩展性和可重构性，并且更好地支持在不同端口上进行实时通信和数据交互。\n'

In [27]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores.faiss import FAISS

loader = WebBaseLoader("http://www.paulgraham.com/greatwork.html")
docs = loader.load()

# split the page into chunks (uses RecursiveCharacterTextSplitter already present in the notebook)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

# build a FAISS vectorstore using the existing OllamaEmbeddings class in the notebook
vs = FAISS.from_documents(chunks, OllamaEmbeddings(model="llama2-chinese:13b", base_url="http://localhost:22515"))

# run a similarity search as a replacement for VectorstoreIndexCreator().query(...)
vs.similarity_search("What should I work on?")

KeyboardInterrupt: 

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# 初始化稀疏匹配的检索器
bm25_retriever = BM25Retriever.from_texts(doc_list)
bm25_retriever.k = 2

# 初始化密集匹配的检索器
embedding = OpenAIEmbeddings()
faiss_vectorstore = FAISS.from_texts(doc_list, embedding)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs = {"k": 2})

# 初始化 EnsembleRetriever 检索器：传入两个子检索器和它们的融合权重
ensemble_retriever = EnsembleRetriever(
    retrievers = [bm25_retriever, faiss_retriever],
    weights = [0.5, 0.5]
)

docs = ensemble_retriever.get_relevant_documents("<raw question here>")


KeyboardInterrupt



In [2]:
from langchain_core.runnables import RunnableLambda
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores.faiss import FAISS
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# 通过文档加载器正常加载文档并通过文本分割器进行分割
documents = TextLoader("/home/fairywell/ml/LangChainPractice/Chapter5/example_data/12百家讲坛/百家讲坛.txt.ok").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

# 基于 FAISS 向量存储构建基础检索器
faiss_vs = FAISS.from_documents(texts, OllamaEmbeddings(model="llama2-chinese:13b", base_url="http://localhost:22515"))
retriever = faiss_vs.as_retriever()

# VectorStoreRetriever may not expose get_relevant_documents directly in some langchain versions.
# Wrap the underlying vectorstore to provide the get_relevant_documents API expected by other components.
class SimpleRetriever:
    def __init__(self, vectorstore, search_kwargs=None):
        self.vectorstore = vectorstore
        self.search_kwargs = search_kwargs or {}

    def get_relevant_documents(self, query):
        return self.vectorstore.similarity_search(query, **self.search_kwargs)

base_retriever = SimpleRetriever(retriever.vectorstore, getattr(retriever, "search_kwargs", {}))
# 初始文档可以通过基础检索器获取
docs = base_retriever.get_relevant_documents("中国诗词")

# Wrap the simple retriever in a Runnable so ContextualCompressionRetriever accepts it
base_retriever_runnable = RunnableLambda(lambda query, **kwargs: base_retriever.get_relevant_documents(query))

# 基于 OpenAI 能力构建一个文档压缩器，它将逐一处理初始文档并从每个文档中提取与查询最相关的部分
llm = ChatOllama(model="llama2-chinese:13b", base_url="http://localhost:22515")
compressor = LLMChainExtractor.from_llm(llm)

# 最后把基础检索器和文档压缩器传入 ContextualCompressionRetriever 让它进行问答的检索，对上下文进行压缩并输出结果
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever_runnable
)

try:
    compressed_docs = compression_retriever.get_relevant_documents("百家讲坛")
    print(compressed_docs)
except Exception as e:
    print("Error during compression retrieval:", e)
    # Debug: print the first document and query
    if texts:
        print("First document content:", texts[0].page_content[:500])
    else:
        print("No texts available.")

Error during compression retrieval: 'ContextualCompressionRetriever' object has no attribute 'get_relevant_documents'
First document content: 百家讲坛-历史上的和珅 
作者: 纪连海 
简介：
    我们现在的影视节目中播出的清宫戏，大多数演的都是康乾盛世时期的事。在这些作品中，和珅是所有描写关于乾隆时代的影视作品中的第一重要角色。但是，细心的观众通过这些各具特色的关于和珅的表演就会发现，这些影视作品中的和珅居然有着两种完全不同的角色：一种是在《宰相刘罗锅》和《铁齿铜牙纪晓岚》中由著名演员王刚主演的和珅，是一个完全的反面角色：他靠溜须拍马为生，不但自私，而且极端贪婪。... 
什么样的家庭造就了和珅？

楔子：三个完全不同的和珅

    我们现在的影视节目中播出的清宫戏，大多数演的都是康乾盛世时期的事。在这些作品中，和珅是所有描写关于乾隆时代的影视作品中的第一重要角色。 
    但是，细心的观众通过这些各具特色的关于和珅的表演就会发现，这些影视作品中的和珅居然有着两种完全不同的角色： 

    一种是在《宰相刘罗锅》和《铁齿铜牙纪晓岚》中由著名演员王刚主演的和珅，是一个完全的反面角色：他靠溜须拍马为生，不但自私，而且极端贪婪。 

    另一种是在《乾隆王朝》中由演员陈锐主演的和珅，是一个完全的正面角色：他忧国忧民


/tmp/ipykernel_5845/3271431255.py:39: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama2-chinese:13b", base_url="http://localhost:22515")


In [3]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores.chroma import Chroma
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_community.vectorstores.utils import filter_complex_metadata
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# 设置 OpenAI API Key（从环境或交互式输入获取）
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 修复：避免把错误的字符串（例如误把 base_url=... 设为 API key）当作 key 使用
if OPENAI_API_KEY and "base_url" in OPENAI_API_KEY:
    print("Detected malformed OPENAI_API_KEY in environment; clearing it.")
    OPENAI_API_KEY = None

if not OPENAI_API_KEY:
    # 在 Jupyter 中安全地输入 API key（不回显）
    try:
        from getpass import getpass

        OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")
    except Exception:
        OPENAI_API_KEY = input("Enter your OpenAI API key: ")

if not OPENAI_API_KEY:
    raise ValueError("OpenAI API key is required. Set OPENAI_API_KEY env var or input it when prompted.")

# 确保环境变量被设置，以便 ChatOpenAI 等也能读取到
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# 准备一些实验用的数据，请重点关注 metadata 元数据部分的内容
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams" \
        "within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

# 基于 Chroma 向量存储构建基础检索器
try:
    # 使用可用的嵌入模型（例如 text-embedding-3-small），避免使用已下线的 text-embedding-ada-002
    vectorstore = Chroma.from_documents(
        filter_complex_metadata(docs),
        OpenAIEmbeddings(
            model="text-embedding-3-small",
            api_key=OPENAI_API_KEY,
            base_url="https://api.lingyaai.cn/v1/",
        ),
    )
except Exception as e:
    print("Error creating Chroma vectorstore / embeddings:", e)
    raise

# 【重要】定义在自组织查询中用于提取结构化数据的数据结构（细化到属性名称、属性描述、类型）
# NOTE: use a generic numeric type ("number") so numeric comparisons are handled as numeric
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        # use a generic numeric type so translators treat comparisons as numeric
        type="int",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]


# 提供文档主体内容的描述（也是结构化数据的一部分）
document_content_description = "Brief summary of a movie"

# 构建 SelfQueryRetriever 检索器：把以上准备的大语言模型、向量存储、结构化数据描述一并传入
retriever = SelfQueryRetriever.from_llm(
    llm=ChatOpenAI(model_name="gpt-4.1-mini", base_url="https://api.lingyaai.cn/v1/", temperature=0),
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    # 通过这个参数让检索器可以识别自然语言定义的文档返回数量
    enable_limit=True,
)

#retriever.invoke("What are the comedy movies in the list? Return up to 2 results.")
#retriever.invoke("I want to watch a movie directed by Christopher Nolan. What are my options?")
retriever.invoke("What is a scientific fiction movie in the list?")
retriever.invoke("What's a movie after 1990 but before 2005 that's all about toys, and preferably is animated?")
#retriever.invoke("I want to watch a movie rated higher than 8")

ValueError: Expected operand value to be an int or a float for operator $gt, got 1990 in query.

In [ ]:
from datetime import datetime, timedelta
import faiss

from langchain_core.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore import InMemoryDocstore
from langchain.retrievers import TimeWeightedVectorStoreRetriever

# 初始化向量存储
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small", base_url="https://api.lingyaai.cn/v1/", api_key=OPENAI_API_KEY)
embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embedding_function=embeddings_model.embed_query, index=index, docstore=InMemoryDocstore({}), index_to_docstore_id={})

# 初始化 TimeWeightedVectorStoreRetriever，把衰减率设置得极低（接近0, 0表示永不衰减）
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.0000000000000000001, # 设置一个非常低的衰减率
    k=1
)   

# 为所有 Document 对象添加时间戳元数据
yesterday = datetime.now() - timedelta(days=1)
retriever.add_documents([
    Document(page_content="hello world", metadata={"last_accessed_at": yesterday})
])
retriever.add_documents([
    Document(page_content="hello foo")
])

retriever.get_relevant_documents("hello")

ModuleNotFoundError: No module named 'langchain.schema'

In [1]:
# 5.5 父文档回溯
import os
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# 设置 OpenAI API Key（从环境或交互式输入获取）
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 修复：避免把错误的字符串（例如误把 base_url=... 设为 API key）当作 key 使用
if OPENAI_API_KEY and "base_url" in OPENAI_API_KEY:
    print("Detected malformed OPENAI_API_KEY in environment; clearing it.")
    OPENAI_API_KEY = None

if not OPENAI_API_KEY:
    # 在 Jupyter 中安全地输入 API key（不回显）
    try:
        from getpass import getpass

        OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")
    except Exception:
        OPENAI_API_KEY = input("Enter your OpenAI API key: ")

if not OPENAI_API_KEY:
    raise ValueError("OpenAI API key is required. Set OPENAI_API_KEY env var or input it when prompted.")

# 确保环境变量被设置，以便 ChatOpenAI 等也能读取到
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# 向量存储用于存储小块文档及其文本向量表示
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small", base_url="https://api.lingyaai.cn/v1/", api_key=OPENAI_API_KEY))  
# 普通存储用于存储大块文档，这里使用内存作为普通存储
storage = InMemoryStore()

# 父文档分割器用于分割大块文档
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# 子文档分割器用于分割小块文档（文本片段的粒度需要小于父文档分割器分割后的文本片段粒度）
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

# 构造 ParentDocumentRetriever 检索器
from langchain_classic.retrievers.parent_document_retriever import ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=storage,
    parent_splitter=parent_splitter,
    child_splitter=child_splitter
)

# ParentDocumentRetriever 构建完成后，可以直接添加文档、建立索引，后续文本分割和向量化都在其内部完成
retriever.add_documents([
    Document(page_content="中国人是华夏名族的统称，我们生活在东方大陆的长江中下游平原地区。", metadata={"source": "doc1"})
])
retriever.add_documents([
    Document(page_content="日本是一个有着悠久历史和文化的国家，位于东亚海洋地区。", metadata={"source": "doc2"})
])

# 使用 ParentDocumentRetriever 进行检索：兼容多种 retriever API（部分版本没有 get_relevant_documents）
# 很好的一个包装 get_relevant_documents() 的示例
query = "中国人"
if hasattr(retriever, "get_relevant_documents"):
    retrieved_docs = retriever.get_relevant_documents(query)
elif hasattr(retriever, "retrieve"):
    retrieved_docs = retriever.retrieve(query)
elif callable(retriever):
    # some retrievers implement __call__
    retrieved_docs = retriever(query)
else:
    # 最后尝试访问底层 vectorstore 的相似度搜索
    try:
        underlying_vs = getattr(retriever, "vectorstore", None)
        if underlying_vs and hasattr(underlying_vs, "similarity_search"):
            retrieved_docs = underlying_vs.similarity_search(query)
        else:
            raise AttributeError("No compatible retrieval method found on retriever")
    except Exception as e:
        print("Retrieval error:", e)
        retrieved_docs = []

print(retrieved_docs)

[Document(id='73b03121-ef7e-4d69-bee5-410ef2511433', metadata={'source': 'doc1', 'doc_id': 'c875e11f-495c-40ee-9715-3311a11e5b46'}, page_content='中国人是华夏名族的统称，我们生活在东方大陆的长江中下游平原地区。'), Document(id='8841ac95-2658-4928-ae9f-caedf8dfb3fe', metadata={'source': 'doc2', 'doc_id': '3dce7334-40dc-4386-afe4-946307816615'}, page_content='日本是一个有着悠久历史和文化的国家，位于东亚海洋地区。')]
